# 3. Pull Census households by county

Downloads, for every county, the number of households and the number of people living in households, from the Census Bureau's American Community Survey (5-year estimates). Notebook 5 uses these to estimate people per household in each utility's service area.

| Table | What it is |
|---|---|
| B25003 | Occupied housing units, i.e. households |
| B25008 | Population living in occupied housing units |

These come from the Census "table-based summary files", which need no API key (the Census API now does). The national files are large, so only the county rows are kept, saved to `data/raw/census/YYYY/county_households.csv`. Skipped if that year is already there.

Note: since 2022 the Census reports Connecticut by its nine planning regions instead of counties.

**Parameter:** `acs_year` (defaults to `ACS_YEAR` in `common.py`).

In [ ]:
acs_year = None

In [ ]:
import io

import pandas as pd
import requests

from common import ACS_YEAR, RAW_CENSUS_DIR, get_logger

log = get_logger("pull_census")
acs_year = acs_year or ACS_YEAR

BASE = f"https://www2.census.gov/programs-surveys/acs/summary_file/{acs_year}/table-based-SF"
COUNTY = "0500000US"  # GEO_ID prefix for counties

In [ ]:
def county_rows(url, usecols):
    """Download a pipe-delimited summary file and keep the header plus county rows."""
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    # The server doesn't declare a charset; the files are UTF-8 (e.g. "Doña Ana").
    lines = resp.content.decode("utf-8").splitlines()
    gid = lines[0].split("|").index("GEO_ID")
    keep = [lines[0]] + [line for line in lines[1:] if line.split("|")[gid].startswith(COUNTY)]
    return pd.read_csv(io.StringIO("\n".join(keep)), sep="|", usecols=usecols, dtype=str)


out_dir = RAW_CENSUS_DIR / str(acs_year)
out_csv = out_dir / "county_households.csv"
if out_csv.exists():
    print(f"ACS {acs_year} county households already in {out_csv}")
else:
    geos = county_rows(f"{BASE}/documentation/Geos{acs_year}5YR.txt", ["STUSAB", "GEO_ID", "NAME"])
    households = county_rows(f"{BASE}/data/5YRData/acsdt5y{acs_year}-b25003.dat", ["GEO_ID", "B25003_E001"])
    population = county_rows(f"{BASE}/data/5YRData/acsdt5y{acs_year}-b25008.dat", ["GEO_ID", "B25008_E001"])
    counties = (geos.merge(households, on="GEO_ID").merge(population, on="GEO_ID")
                .rename(columns={"STUSAB": "state", "NAME": "county_name",
                                 "B25003_E001": "households", "B25008_E001": "household_population"}))
    counties[["households", "household_population"]] = counties[["households", "household_population"]].astype(int)
    out_dir.mkdir(parents=True, exist_ok=True)
    counties.to_csv(out_csv, index=False)
    log.info("downloaded ACS %s 5-year county households: %d counties, %s households, %s people in households",
             acs_year, len(counties), f"{counties['households'].sum():,}", f"{counties['household_population'].sum():,}")

pd.read_csv(out_csv).head()